In [1]:
import pandas as pd
import numpy as np

Data = pd.read_csv('messy_student_records.csv')

In [2]:
# Data quality report
print(f"Total Rows & Columns: {Data.shape}")
print(f"Data Quality Report\n")
print(f"Count of Null Values: \n{Data.isna().sum()}")
print(f"\nNo. of Duplicate Rows: \n{Data.duplicated().sum()}")
print(f"\nData Types: \n{Data.dtypes}")
print(f"\nValue Range Anomalies: \nStudent_ID {Data.describe()}")

Total Rows & Columns: (1555, 12)
Data Quality Report

Count of Null Values: 
Student_ID           0
Student_Name        16
Age                  3
Gender               0
Department          16
Phone              289
Email              140
Enrollment_Date     90
GPA                 82
Attendance_%        79
Tuition_Fees        55
Status               0
dtype: int64

No. of Duplicate Rows: 
34

Data Types: 
Student_ID         object
Student_Name       object
Age                object
Gender             object
Department         object
Phone              object
Email              object
Enrollment_Date    object
GPA                object
Attendance_%       object
Tuition_Fees       object
Status             object
dtype: object

Value Range Anomalies: 
Student_ID        Student_ID Student_Name   Age  Gender        Department Phone  \
count        1555         1539  1552    1555              1539  1266   
unique       1472          587    37      11                32  1224   
top      STU-0

In [3]:
Data.head()

,Student_ID,Student_Name,Age,Gender,Department,Phone,Email,Enrollment_Date,GPA,Attendance_%,Tuition_Fees,Status
0,STU-0481,Grace Williams,34,Male,Economics,(080)32768466,grace.williams405@example.com,22/03/2025,2.41,64.7,824691.64,graduated
1,STU-0002,Sarah James,32,FEMALE,Computer Science,(080)47518305,NaN,2022-10-19,3.77,81.2,826197.64,Actve
2,STU-0003,SARAH ADEBAYO,38,FEMALE,MASS COMM.,08092157763,sarah.adebayo208@example.com,10-31-2023,3.18,89.8,919574.89,suspended
3,STU-0004,Chiamaka Garba,30,MALE,business admin,NaN,chiamaka.garba204@example.com,NaN,3.3,86.3,787034.87,Active
4,STU-0005,Chiamaka Adebayo,34,Femle,computer science,NaN,chiamaka.adebayo574@example.com,17/08/2020,2.36,85.5,789833.7,ACTIVE


#### Student Name

In [4]:
# Count missing value
print(Data['Student_Name'].isna().sum())

16


In [5]:
Data['Student_Name'].unique()[:20]

array(['  Grace Williams  ', 'Sarah James', 'SARAH ADEBAYO',
       '   Chiamaka   Garba   ', 'Chiamaka Adebayo', 'David Okafor',
       'Mary Adebayo', 'Ibrahim Adeyemi', ' Ibrahim   Brown ',
       'SAMUEL SMITH', 'Amina Adeyemi', 'Amina Johnson', ' Mary   Bello ',
       'Samuel Garba', 'Fatima Bello', ' Daniel   Smith ',
       ' Amina   Usman ', 'MICHAEL THOMAS', 'David Usman', 'Sarah Okafor'],
      dtype=object)

In [6]:
# Convert all to lowercase and strip whitespace
Data['Student_Name'] = Data['Student_Name'].astype(str).str.lower().str.strip().str.replace(r'\s+', ' ', regex=True)

In [7]:
mask = Data['Student_Name'] == '-'
Data[mask]

,Student_ID,Student_Name,Age,Gender,Department,Phone,Email,Enrollment_Date,GPA,Attendance_%,Tuition_Fees,Status
906,STU-0907,-,41,MALE,MASS COMM.,08011128094,sarah.williams425@example,09-27-2022,2.16,81.3,1128905.25,suspended
1194,STU-1195,-,21,Femle,mass communication,+234 8023856295,chiamaka.eze181@example.com,2022-03-29,2.51,64.9,645648.6,Active


In [8]:
# Replace nan and - with UNKNOWN
Data['Student_Name'] = Data['Student_Name'].replace(['-', 'nan'], 'UNKOWN')
Data['Student_Name'] = Data['Student_Name'].str.title()

In [9]:
print(Data['Student_Name'].unique()[:20])
print(Data['Student_Name'].isna().sum())

['Grace Williams' 'Sarah James' 'Sarah Adebayo' 'Chiamaka Garba'
 'Chiamaka Adebayo' 'David Okafor' 'Mary Adebayo' 'Ibrahim Adeyemi'
 'Ibrahim Brown' 'Samuel Smith' 'Amina Adeyemi' 'Amina Johnson'
 'Mary Bello' 'Samuel Garba' 'Fatima Bello' 'Daniel Smith' 'Amina Usman'
 'Michael Thomas' 'David Usman' 'Sarah Okafor']
0


Student_Name is an identifier field, not a variable used in analysis. Standard imputation methods are inappropriate for names, so missing values were filed with a placeholder to preserve row integrity rather than dropped.

#### Age

In [10]:
# Check Age column for unique entries
Data['Age'].unique()

array(['34', '32', '38', '30', '45', '19', '28', '40', '43', '17', '36',
       '41', '39', '22', '21', '67', '23', '33', '42', '31', '25', '24',
       '27', '12', '37', '18', '44', '29', '35', '20', '15', '26', '72',
       'unknown', 'not available', '?', nan, '99'], dtype=object)

In [11]:
# Convert all text values to numberic and force errors to NaN
print(f"Before - missing values: {Data['Age'].isna().sum()}")

Data['Age'] = pd.to_numeric(Data['Age'], errors='coerce')
Data['Age'] = Data['Age'].replace(99, np.nan)

print(Data['Age'].unique())
print(f"\nAfter - missing values: {Data['Age'].isna().sum()}")

Before - missing values: 3
[34. 32. 38. 30. 45. 19. 28. 40. 43. 17. 36. 41. 39. 22. 21. 67. 23. 33.
 42. 31. 25. 24. 27. 12. 37. 18. 44. 29. 35. 20. 15. 26. 72. nan]

After - missing values: 15


In [12]:
Data['Age'].describe()

count    1540.000000
mean       31.491558
std         9.172978
min        12.000000
25%        24.000000
50%        32.000000
75%        39.000000
max        72.000000
Name: Age, dtype: float64

In [13]:
# Flag statistical outliers using IQR
Q1 = Data['Age'].quantile(0.25)
Q3 = Data['Age'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = Data[(Data['Age'] < lower) | (Data['Age'] > upper)]
print(outliers[['Student_ID', 'Student_Name', 'Age']])

     Student_ID    Student_Name   Age
18     STU-0281     David Usman  67.0
205    STU-0206     Esther Musa  67.0
306    STU-0307      Daniel Obi  72.0
478    STU-0479    Esther Garba  72.0
553    STU-0554      Mary Brown  72.0
636    STU-0637    Peter Okafor  67.0
712    STU-0713       David Obi  67.0
1208   STU-1209      John Usman  67.0
1394   STU-1395   David Adeyemi  67.0
1477   STU-1478  Ibrahim Okafor  72.0
1502   STU-0713       David Obi  67.0


In [14]:
# Fill missing rows in Age col with the median age
Data['Age'] = Data['Age'].fillna(Data['Age'].median())

print(f"Final: missing values: {Data['Age'].isna().sum()}")

Final: missing values: 0


Non-numeric entries ('unknown', 'not available', '?') were converted to NaN using pd.to_numeric(errors='coerce'), bring missing values to 15. The age 72 was checked against IQR bounds, judged plausible,and retained. Remaining missing values were filled with the median (≈32), chosen over the mean due to the column's skew.

#### Gender

In [15]:
print(Data['Gender'].unique())
print(f"Missing values: {Data['Gender'].isna().sum()}")

['Male' 'FEMALE' 'MALE' 'Femle' 'male' 'F' 'mle' 'M' 'Female' 'femal'
 'female']
Missing values: 0


In [16]:
# Convert all to lowercase and replace values
Data['Gender'] = Data['Gender'].astype(str).str.lower()

Data['Gender'] = Data['Gender'].replace(['male', 'm', 'mle'], 'Male')
Data['Gender'] = Data['Gender'].replace(['female', 'f', 'femle', 'femal'], 'Female')

print(Data['Gender'].unique())

['Male' 'Female']


#### Department

In [17]:
# Check for missing values and unique entries
print(Data['Department'].unique())
print(f"Missing values: {Data['Department'].isna().sum()}")

['Economics' 'Computer Science' 'MASS COMM.' 'business admin'
 'computer science' 'Bio chemistry' 'BUSINESS ADMIN' 'COMPUTER SCIENCE'
 '  BIOCHEMISTRY  ' 'ECONOMICS' 'mass communication' 'BIOCHEMISTRY'
 'Mass Communication ' 'Business Admin' 'economics' 'Mass Communication'
 'Biochemistry' 'ComputerScience' 'Business Administration'
 '  ComputerScience  ' '  Mass Communication  ' nan '  Biochemistry  '
 '  Mass Communication   ' '-' '  computer science  '
 '  Business Administration  ' '  Computer Science  ' '  Economics  '
 '  Business Admin  ' '  Bio chemistry  ' '  MASS COMM.  '
 '  mass communication  ']
Missing values: 16


In [18]:
# Convert all to lowercase and strip whitespaces
Data['Department'] = Data['Department'].astype(str).str.lower().str.strip()

In [19]:
# Replace values
Data['Department'] = Data['Department'].replace('economics', 'Economics')
Data['Department'] = Data['Department'].replace(['computer science', 'computerscience'], 'Computer Science')
Data['Department'] = Data['Department'].replace(['mass comm.', 'mass communication'], 'Mass Communication')
Data['Department'] = Data['Department'].replace(['business admin', 'business administration'], 'Business Administration')
Data['Department'] = Data['Department'].replace(['bio chemistry', 'biochemistry'], 'Bio Chemistry')
Data['Department'] = Data['Department'].replace(['-', 'nan'], 'UNKNOWN')

print(Data['Department'].unique())

['Economics' 'Computer Science' 'Mass Communication'
 'Business Administration' 'Bio Chemistry' 'UNKNOWN']


In [20]:
Data['Department'].isna().sum()

0

The Department column contained inconsistent casing and formatting (e.g., 'MASS COMM.', 'business admin', 'ComputerScience'), as well as missing values represented as NaN and '-' strings. All entries were converted to lowercase and stripped of leading/trailing whitespace, then mapped to a consistent set of standardized labels (e.g., 'mass comm.' and 'mass communication' → 'Mass Communication'). Missing and placeholder values (NaN, '-') strings were replaced with 'UNKNOWN' rather than imputed, since department is a categorical identifier and guessing a specific value would misrepresent the student's actual program.

### Phone

In [21]:
# Check for weird phone number formatting and missing values
print(Data['Phone'].unique())
print(f"Missing values: {Data['Phone'].isna().sum()}")

['(080)32768466' '(080)47518305' '08092157763' ... '234833041068'
 '(080)39053522' '234959536173']
Missing values: 289


In [22]:
# Standardize the format - strip all non-digits and normalize to a single format
import re

def  clean_phone(phone):
    if pd.isna(phone):
        return phone
    digits = re.sub(r'\D', '', str(phone))   # strip non-digits
    if digits.startswith('234'):
        digits = '0' + digits[3:]       # convert 234xxx -> 0xxx
    return digits

Data['Phone'] = Data['Phone'].apply(clean_phone)

In [23]:
Data['Phone'].unique()

array(['08032768466', '08047518305', '08092157763', ..., '0833041068',
       '08039053522', '0959536173'], dtype=object)

In [24]:
# Handle missing values
Data['Phone'] = Data['Phone'].fillna('Not Provided')

In [25]:
Data['Phone'].isna().sum()

0

In [26]:
# Flag any number that aren't 11 digits
invalid = Data[Data['Phone'].str.len() != 11]
print(invalid[['Student_ID', 'Phone']])

     Student_ID         Phone
3      STU-0004  Not Provided
4      STU-0005  Not Provided
14     STU-0015    0929889096
17     STU-0018  Not Provided
18     STU-0281  Not Provided
...         ...           ...
1543   STU-0873    0818960575
1546   STU-0201  Not Provided
1547   STU-0061    0850156836
1548   STU-0587  Not Provided
1551   STU-1390    0959536173

[514 rows x 2 columns]


In [27]:
# Flag invalid (wrong-length) numbers seperately from missing/placeholder ones
Data.loc[(Data['Phone'] != 'Not Provided') & (Data['Phone'].str.len() != 11), 'Phone'] = 'Invalid'

Phone numbers were standardized by stripping non-digit characters and normalizing the '234' country-code prefix to a leading '0'. Missing values were filled with 'Not Provided'. Numbers that did not resolve to the expected 11-digits format after cleaning were flagged as 'Invalid' rather than merged with missing values, since these likely stem from data entry errors rather than absent data.

#### Email

In [28]:
print(Data['Email'].unique())
print(f"Missing values: {Data['Email'].isna().sum()}")

['grace.williams405@example.com' nan ' sarah.adebayo208@example.com ' ...
 'fatima.james937@example.com' 'amina.bello507@example.com'
 'emeka.obi77@example.com']
Missing values: 140


In [29]:
# Strip whitespace
Data['Email'] = Data['Email'].astype(str).str.strip().str.lower()
Data['Email'] = Data['Email'].replace('nan', np.nan)

In [30]:
# Check for entries that don't look like real emails
invalid_email = Data[~Data['Email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', na=False)]
print(invalid_email[['Student_ID', 'Email']])

     Student_ID                     Email
1      STU-0002                       NaN
6      STU-0007                       NaN
14     STU-0015                       NaN
24     STU-0025   john.adebayo840@example
29     STU-0030  ibrahim.brown740@example
...         ...                       ...
1524   STU-1406                       NaN
1535   STU-1051                       NaN
1540   STU-0694                       NaN
1546   STU-0201                       NaN
1550   STU-0716                       NaN

[219 rows x 2 columns]


In [31]:
# Split into missing vs malformed
missing_email = Data[Data['Email'].isna()]
malformed_email = Data[Data['Email'].notna() & ~Data['Email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', na=False)]

print(f"Missing: {len(missing_email)}, Malformed: {len(malformed_email)}")
print(malformed_email[['Student_ID', 'Email']])

Missing: 140, Malformed: 79
     Student_ID                       Email
24     STU-0025     john.adebayo840@example
29     STU-0030    ibrahim.brown740@example
34     STU-0035    emeka.adebayo380@example
64     STU-0065    samuel.ibrahim80@example
109    STU-0110  ibrahim.adeyemi685@example
...         ...                         ...
1457   STU-1458     fatima.smith742@example
1460   STU-1461      john.okafor444@example
1475   STU-1476     mary.adeyemi150@example
1490   STU-1491       daniel.eze775@example
1515   STU-0829     esther.brown168@example

[79 rows x 2 columns]


In [32]:
mask1 = Data['Email'].notna() & Data['Email'].str.match(r'^[\w\.-]+@example$')
Data.loc[mask1, 'Email'] = Data.loc[mask1, 'Email'] + '.com'

In [33]:
# Re-check AFTER applying the fix
malformed_email = Data[Data['Email'].notna() & ~Data['Email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', na=False)]

print(f'Malformed email count: {len(malformed_email)}')
print(malformed_email[['Student_ID', 'Email']])

Malformed email count: 2
     Student_ID Email
977    STU-0978     -
1207   STU-1208     -


In [34]:
# Fill missing values and repalce '-' 
Data['Email'] = Data['Email'].fillna('Not Provided')
Data['Email'] = Data['Email'].replace('-','Not Provided')

print(f"Missing value: {Data['Email'].isna().sum()}")

Missing value: 0


The Email column contained leading/trailing whitespace, missing values represented as NaN, and 79 malformed addresses missing the .com domain suffix (e.g., name@example). Whitespace was stripped and all text lowercased for consistency. Malformed addresses following the @example pattern were corrected by appending .com, since the error wassystematic and unambiguous to fix. Two remaining entries stored as '-' were treated as missing rather than malformed . Since email is a unique identifier and not a value that can be reasonably imputed, all missing/placeholder entries were filled with 'Not Provided' rather than guessed.

#### Enrollment_Date

In [35]:
print(Data['Enrollment_Date']. unique())
print(f"Missing value: {Data['Enrollment_Date'].isna().sum()}")

['22/03/2025' '2022-10-19' '10-31-2023' ... '12-12-2022' '10-19-2024'
 '2023-01-28']
Missing value: 90


In [36]:
# Handle inconsistent date format

def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    if val == '-':
        return pd.NaT
    for fmt in ('%d/%m/%Y', '%Y-%m-%d', '%m-%d-%Y', '%b %d, %Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except ValueError:
            continue
    return pd.NaT

Data['Enrollment_Date'] = Data['Enrollment_Date'].apply(parse_date)
print(f"Missing after parsing: {Data['Enrollment_Date'].isna().sum()}")

Missing after parsing: 91


The Enrollment_Date column contained four inconsistent date formats mixed together - DD/MM/YYYY, YYYY-MM-DD, MM-DD-YYYY, and Mon DD, YYYY (e.g., 'Jan 06, 2023'), along with a placeholder '-' and true missing values (90 total). A custom parsing function was used to attempt each known format in sequence and convert values to proper datetime objects, since a single pd.to_datetime() call could not reliably handle multiple mixed formats without risking misread day/month values.

The '-' placeholder and genuinely missing entries (91 total) were left as NaT rather than filled with an imputed or placeholder date, since a fabricated entrollment date would misrepresent an event-specific value. This preserves the column's datetime dtype for any future date-based analysis.

#### GPA

In [37]:
print(Data['GPA'].unique())
print(f"Before - missing values: {Data['GPA'].isna().sum()}")

['2.41' '3.77' '3.18' '3.3' '2.36' '3.73' '3.32' '2.51' '2.96' '2.44'
 '3.12' '2.88' nan '3.05' '3.39' '4.13' '3.43' '9.0' '3.89' '3.06' '3.65'
 '2.63' '1.97' '3.63' '2.95' '2.35' '2.52' '3.0' '2.86' '2.5' '2.53'
 '2.99' '3.01' '2.98' '2.82' '3.07' '3.25' '2.33' '3.04' '2.81' '2.3'
 '2.37' '2.32' '2.4' '3.5' '3.23' '1.22' '2.83' '3.7' '2.38' '2.14' '3.79'
 '3.38' '2.28' '2.85' '3.92' '3.15' '2.97' '2.58' '3.67' '2.74' '2.77'
 '2.07' '4.24' '2.54' '1.99' '2.15' '1.96' '3.82' '3.55' '2.84' '2.75'
 '3.42' '3.2' '3.44' '2.19' '2.64' '3.31' '4.42' '3.29' '3.51' '4.07'
 '2.67' '2.59' '2.92' '3.02' '2.68' '2.8' '3.21' '3.1' '3.66' '3.13'
 'unknown' '-1.0' '2.24' '2.65' '5.5' '2.22' '3.96' '3.93' '?' '3.27'
 '4.8' '2.9' '3.17' '3.4' '3.45' '2.55' '2.25' '4.05' '4.06' '3.19' '2.45'
 '2.78' '1.74' '2.89' '2.43' '2.42' '2.72' '2.91' '3.54' '1.77' '3.48'
 '3.11' '3.97' '3.72' '2.09' '2.71' '2.7' '3.46' '3.57' '0.0' '3.74'
 '2.02' '3.16' '2.87' '3.75' '3.36' '3.6' '3.28' '1.93' '3.84' '2.1'
 '2.13'

In [38]:
Data['GPA'] = pd.to_numeric(Data['GPA'], errors='coerce')

print(f"After coercion - missing values: {Data['GPA'].isna().sum()}")
print(Data['GPA'].describe())

After coercion - missing values: 91
count    1464.000000
mean        2.990635
std         0.869902
min        -1.000000
25%         2.600000
50%         2.980000
75%         3.380000
max         9.000000
Name: GPA, dtype: float64


In [39]:
# Assuming a 0.0 - 5.0 scale 
invalid_gpa = Data[(Data['GPA'] < 0) | (Data['GPA'] > 5)]
print(invalid_gpa[['Student_ID', 'GPA']])

     Student_ID  GPA
19     STU-0708  9.0
132    STU-0133 -1.0
137    STU-0138  5.5
222    STU-0223  5.5
226    STU-0227  5.5
256    STU-0257 -1.0
377    STU-0378  9.0
439    STU-0440 -1.0
448    STU-0449  5.5
489    STU-0490  5.5
556    STU-0557  5.5
573    STU-0574 -1.0
614    STU-0615  5.5
629    STU-0630  5.5
632    STU-0633  9.0
636    STU-0637  9.0
663    STU-0664  9.0
664    STU-0665 -1.0
706    STU-0707  5.5
731    STU-0732  9.0
784    STU-0785 -1.0
996    STU-0997 -1.0
1061   STU-1062 -1.0
1133   STU-1134 -1.0
1141   STU-1142  9.0
1187   STU-1188  5.5
1269   STU-1270 -1.0
1290   STU-1291 -1.0
1318   STU-1319  9.0
1388   STU-1389  5.5
1462   STU-1463  5.5
1497   STU-1498 -1.0


In [40]:
# Treat out-of-range GPA values as missing
Data.loc[(Data['GPA'] < 0) | (Data['GPA'] > 5), 'GPA'] = np.nan

print(f"Missing after flagging invalid range: {Data['GPA'].isna().sum()}")

Missing after flagging invalid range: 123


In [41]:
# Impute GPA col with median
Data['GPA'] = Data['GPA'].fillna(Data['GPA'].median())

print(f"After imputation - missing values: {Data['GPA'].isna().sum()}")
print(Data['GPA'].describe())

After imputation - missing values: 0
count    1555.000000
mean        2.970309
std         0.595404
min         0.000000
25%         2.650000
50%         2.980000
75%         3.330000
max         4.800000
Name: GPA, dtype: float64


The GPA column contained non-numeric placeholder entries which were converted to NaN using pd.to_numeric(errors='coerce'). Since Nigerian institutions use a 0.0-5.0 GPA scale, vvalues outside this range were identified as invalid entries rather than genuine GPAs. Because the correct intended values for these out-of-range entries could not be verified, they were treated as missing rather than corrected or guessed. All missing values were then imputed using the median, consistent with the approach used for Age, to avoid distortion from any remaining outliers.

#### Attendance_%

In [42]:
print(Data['Attendance_%'].unique()[:20])
print(f"Before - missing values: {Data['Attendance_%'].isna().sum()}")

['64.7' '81.2' '89.8' '86.3' '85.5' '90.2' '72.7' '83.8' '83.2' '64.1'
 '0.0' '92.8' '65.5' '91.7' '76.8' '69.8' '80.8' '64.3' '87.6' nan]
Before - missing values: 79


In [43]:
# Convert to numeric values
Data['Attendance_%'] = pd.to_numeric(Data['Attendance_%'], errors='coerce')

print(Data['Attendance_%'].unique()[:20])
print(f"\nAfter - missing values: {Data['Attendance_%'].isna().sum()}")

[64.7 81.2 89.8 86.3 85.5 90.2 72.7 83.8 83.2 64.1  0.  92.8 65.5 91.7
 76.8 69.8 80.8 64.3 87.6  nan]

After - missing values: 88


In [44]:
# Check for negatives or anything above 100
invalid_attendance = Data[(Data['Attendance_%'] < 0) | (Data['Attendance_%'] > 100)]
print(invalid_attendance[['Student_ID', 'Attendance_%']])

     Student_ID  Attendance_%
38     STU-0039         100.2
43     STU-0044         103.1
61     STU-0062         115.9
63     STU-0064         130.0
71     STU-0072         100.2
...         ...           ...
1525   STU-0644         130.0
1529   STU-0064         130.0
1536   STU-1316         110.0
1540   STU-0694         103.5
1554   STU-0879         106.6

[150 rows x 2 columns]


In [45]:
# Check for zero attendance
print((Data['Attendance_%'] == 0) .sum())

15


In [46]:
# Treat impossible attendance values as missing
Data.loc[(Data['Attendance_%'] < 0) | (Data['Attendance_%'] > 100), 'Attendance_%'] = np.nan

print(f"Missing after flagging invalid range: {Data['Attendance_%'].isna().sum()}")

Missing after flagging invalid range: 238


In [47]:
Data['Attendance_%'] = Data['Attendance_%'].fillna(Data['Attendance_%'].median())
print(f"After imputation - missing: {Data['Attendance_%'].isna().sum()}")

After imputation - missing: 0


The Attendance_% column contained non-numeric placeholders which were converted to NaN using pd.to_numeric(errors=coerce). Simce attendance is a percentage, values were expected to fall between 0 and 100; however, 150 enteries exceed 100%, indicating a systematic data error rather than isolated typos. Since the correct values for these out-of-range entries could not be verified, they were treated as missing rather than corrected or guessed. Zero values were retained as valid, since 0% attendance is a plausible extreme rather than a impossible one. All missing values were imputed using the median, consistent with the approach used for Age and GPA.

#### Tuition_Fees

In [48]:
print(Data['Tuition_Fees'].unique())
print(f"Before - missing values: {Data['Tuition_Fees'].isna().sum()}")

['824691.64' '826197.64' '919574.89' ... '1298851.15' '734009.85'
 '1040600.45']
Before - missing values: 55


In [49]:
# Convert to numeric and check for anomalies
Data['Tuition_Fees'] = pd.to_numeric(Data['Tuition_Fees'], errors='coerce')

print(f"After coercion - missing values: {Data['Tuition_Fees'].isna().sum()}")
print(Data['Tuition_Fees'].describe())

After coercion - missing values: 64
count    1.491000e+03
mean     8.729696e+05
std      3.829700e+05
min     -1.453459e+06
25%      7.607965e+05
50%      8.948557e+05
75%      1.051048e+06
max      1.499348e+06
Name: Tuition_Fees, dtype: float64


In [50]:
# Check for negatives or zero fees
invalid_tuition = Data[(Data['Tuition_Fees'] < 0) | (Data['Tuition_Fees'] == 0)]
print(invalid_tuition[['Student_ID', 'Tuition_Fees']])

     Student_ID  Tuition_Fees
10     STU-0011   -1380559.22
55     STU-0056    -790085.31
58     STU-0059   -1352621.11
135    STU-0136    -703235.11
154    STU-0155    -931866.21
170    STU-0171   -1056267.69
193    STU-0194    -975458.58
217    STU-0218    -700644.38
298    STU-0299   -1408815.95
303    STU-0304   -1173695.73
385    STU-0386    -863130.13
389    STU-0390    -804329.21
398    STU-1408   -1209563.42
514    STU-0515   -1028495.91
552    STU-0553    -643616.48
553    STU-0554    -584115.18
556    STU-0557   -1424741.12
564    STU-0565   -1187848.93
580    STU-0581    -850859.71
606    STU-0607    -756409.23
664    STU-0665   -1083694.14
726    STU-0727    -928254.00
758    STU-0759    -927934.88
815    STU-0816   -1074206.12
848    STU-0849   -1037567.31
880    STU-0881    -956601.97
920    STU-0921    -926659.47
953    STU-0954    -896496.03
980    STU-0981   -1081714.98
1000   STU-1001    -768814.65
1087   STU-1088    -844336.00
1109   STU-1110    -990009.51
1138   STU

In [51]:
# Negative fees are invalid - treat as missing
Data.loc[Data['Tuition_Fees'] < 0, 'Tuition_Fees'] = np.nan

# Flag statistical outliers using IQR
Q1 = Data['Tuition_Fees'].quantile(0.25)
Q3 = Data['Tuition_Fees'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = Data[(Data['Tuition_Fees'] < lower) | (Data['Tuition_Fees'] > upper)]
print(f"Outlier count: {len(outliers)}")
print(outliers[['Student_ID', 'Department', 'Tuition_Fees']])

Outlier count: 9
     Student_ID          Department  Tuition_Fees
113    STU-0114    Computer Science    1479658.78
405    STU-0406    Computer Science    1475398.28
416    STU-0417           Economics    1498693.04
637    STU-0638  Mass Communication    1499348.17
640    STU-0641       Bio Chemistry    1474206.01
736    STU-0737       Bio Chemistry    1473085.73
1129   STU-1130  Mass Communication    1482057.54
1328   STU-1329       Bio Chemistry    1481077.37
1511   STU-0417           Economics    1498693.04


In [52]:
# Fill Tuition_Fees with grouped median (by Department) rather than overall median
Data['Tuition_Fees'] = Data['Tuition_Fees'].fillna(
    Data.groupby('Department')['Tuition_Fees'].transform('median')
)

print(f"After imputation - missing: {Data['Tuition_Fees'].isna().sum()}")

After imputation - missing: 0


The Tuition_Fees column was stored as text and converted to numeric using pd.to_numeric(). Of the flagged outliers, the invalid fees were negative values, which are impossible for a tuition fee rather than statistical outliers, and were treated as missing. The remaining high-end values were judged plausible given department-level-fee variation and were retained. Since tuition fees vary by Department, missing values were imputed using the department-level median rather than a single overall median, to better reflect realistic fee differences across programs.

#### Status

In [53]:
print(Data['Status'].unique())
print(f"Missing values: {Data['Status'].isna().sum()}")

['graduated' 'Actve' 'suspended' 'Active' 'ACTIVE' 'active' 'Suspended'
 'GRADUATED' 'On Leave' 'on leave' 'Graduated' '  Active  ' '  ACTIVE  '
 '  suspended  ' '  Actve  ' '  On Leave  ' '  graduated  '
 '  Suspended  ' '  Graduated  ' '  active  ' '  on leave  '
 '  GRADUATED  ']
Missing values: 0


In [54]:
# Convert all to lowercase, strip whitespaces, and replace
Data['Status'] = Data['Status'].astype(str).str.lower().str.strip()
Data['Status'] = Data['Status'].replace('actve', 'Active')

# Captitalized all values
Data['Status'] = Data['Status'].str.title()

In [55]:
Data['Status'].unique()

array(['Graduated', 'Active', 'Suspended', 'On Leave'], dtype=object)

The Status column contained inconsistent casing ('ACTIVE', 'active', 'Active'), extra whitespace, and one spelling typo ('Actve'). All enties were lowercased, stripped of whitespace, corrected for the typo, and then title-cased for consistentcy, resulting in four clean categories: Graduated, Active, Suspended, and On Leave. No missing values were present in this column, so no imputation was required.

In [56]:
# Check for duplicates rows
print(f"Duplicate rows: {Data.duplicated().sum()}")

Duplicate rows: 43


In [57]:
Data = Data.drop_duplicates()
print(f"Rows after dropping full duplicates: {len(Data)}")

Rows after dropping full duplicates: 1512


In [58]:
# Check for duplicate Student_ID with different other values
dup_ids = Data[Data.duplicated(subset='Student_ID', keep=False)]
print(f"Rows with duplicate Student_ID: {len(dup_ids)}")
print(dup_ids.sort_values('Student_ID').head(10))

Rows with duplicate Student_ID: 80
     Student_ID       Student_Name   Age  Gender               Department  \
24     STU-0025       John Adebayo  41.0  Female         Computer Science   
176    STU-0025  Chiamaka Williams  29.0    Male       Mass Communication   
791    STU-0058     Amina Williams  44.0  Female  Business Administration   
57     STU-0058      Amina Ibrahim  32.0    Male            Bio Chemistry   
72     STU-0073         John Garba  40.0  Female                Economics   
509    STU-0073      Emeka Adebayo  18.0  Female  Business Administration   
1546   STU-0201      Grace Adebayo  31.0  Female  Business Administration   
200    STU-0201      Grace Adebayo  31.0  Female  Business Administration   
351    STU-0250     Michael Thomas  20.0    Male            Bio Chemistry   
249    STU-0250     Blessing Usman  45.0    Male       Mass Communication   

             Phone                             Email Enrollment_Date   GPA  \
24     08087537493       john.adebayo84

In [59]:
# Find IDs where the duplicate rows are actually identical people
true_dupes = Data[Data.duplicated(subset=['Student_ID', 'Student_Name', 'Age', 'Gender'], keep=False)]
print(true_dupes[['Student_ID', 'Student_Name', 'Age']])

     Student_ID     Student_Name   Age
200    STU-0201    Grace Adebayo  31.0
471    STU-0472    Ibrahim Bello  45.0
715    STU-0716   Blessing Bello  37.0
984    STU-0985      Peter Brown  25.0
1050   STU-1051    Fatima Thomas  26.0
1132   STU-1133      Mary Thomas  31.0
1158   STU-1159      John Okafor  40.0
1389   STU-1390  Ibrahim Johnson  39.0
1501   STU-0985      Peter Brown  25.0
1504   STU-0472    Ibrahim Bello  45.0
1507   STU-1159      John Okafor  40.0
1510   STU-1133      Mary Thomas  31.0
1535   STU-1051    Fatima Thomas  26.0
1546   STU-0201    Grace Adebayo  31.0
1550   STU-0716   Blessing Bello  37.0
1551   STU-1390  Ibrahim Johnson  39.0


In [60]:
# Drop those dupes
Data = Data.drop_duplicates(subset=['Student_ID', 'Student_Name', 'Age', 'Gender'], keep='first')

In [61]:
# Identify remaining ID collisions after true duplicates are removed
collisions = Data[Data.duplicated(subset='Student_ID', keep=False)]
print(f"Remaining ID collisions: {len(collisions)}")

# Generate new unique IDs for all but the first occurence of each duplicated ID
def fix_duplicate_ids(df):
    seen =set()
    new_ids = []
    max_num = int(df['Student_ID'].str.extract(r'(\d+)')[0].astype(int).max())
    for id_ in df['Student_ID']:
        if id_ not in seen:
            seen.add(id_)
            new_ids.append(id_)
        else:
            max_num += 1
            new_id = f"STU-{max_num:04d}"
            new_ids.append(new_id)
    return new_ids

Data['Student_ID'] = fix_duplicate_ids(Data)
print(f"Remaining duplicate IDs: {Data['Student_ID'].duplicated().sum()}")

Remaining ID collisions: 64
Remaining duplicate IDs: 0


In [62]:
print(Data.dtypes)

Student_ID                 object
Student_Name               object
Age                       float64
Gender                     object
Department                 object
Phone                      object
Email                      object
Enrollment_Date    datetime64[ns]
GPA                       float64
Attendance_%              float64
Tuition_Fees              float64
Status                     object
dtype: object


In [63]:
summary = pd.DataFrame({
    'Metric': ['Total Rows', 'Duplicate Rows', 'Missing Values (Student_Name)',
               'Missing Values (Age)', 'Missing Values (Department)', 'Missing Values (Phone)', 
               'Missing Values (Email)', 'Missing Values (Enrollment_Date)', 
               'Missing Values (GPA)', 'Missing Values (Attendance_%)',
               'Missing Values (Tuition_Fees)', 'Duplicated Student_ID'],
    'Before': [1555, 34, 16, 3, 16, 289, 140, 90, 82, 79, 55, 80],
    'After': [len(Data), Data.duplicated().sum(), Data['Student_Name'].isna().sum(),
              Data['Age'].isna().sum(), Data['Department'].isna().sum(),
              Data['Phone'].isna().sum(), Data['Email'].isna().sum(),
              Data['Enrollment_Date'].isna().sum(), Data['GPA'].isna().sum(),
              Data['Attendance_%'].isna().sum(), Data['Tuition_Fees'].isna().sum(),
              Data['Student_ID'].duplicated().sum()]
})
print(summary)

                              Metric  Before  After
0                         Total Rows    1555   1504
1                     Duplicate Rows      34      0
2      Missing Values (Student_Name)      16      0
3               Missing Values (Age)       3      0
4        Missing Values (Department)      16      0
5             Missing Values (Phone)     289      0
6             Missing Values (Email)     140      0
7   Missing Values (Enrollment_Date)      90     89
8               Missing Values (GPA)      82      0
9      Missing Values (Attendance_%)      79      0
10     Missing Values (Tuition_Fees)      55      0
11             Duplicated Student_ID      80      0


In [64]:
# Save the cleaned dataset
Data.to_csv('cleaned_student_records.csv', index=False)
print('Saved successfully.')

Saved successfully.
